# Comprehensive Model Training — Google Colab GPU Version

Trains ALL models for MetaTrader Multi-Strategy EA on **Colab GPU (T4/A100)**.

## Models
- **PyTorch (ONNX)**: SequenceMLP, PatchTST, iTransformer
- **Tree-Based**: Forex(LGBM), Metals(Cat+XGB), Indices(XGB), Energies(XGB)
- **Deriv Families (18)**: CatBoost/LGBM/XGBoost per family
- **Stackers**: Ridge on OOF predictions

## Colab Setup
1. **Runtime → Change runtime type → GPU** (T4 free, A100 Pro)
2. **Mount Google Drive** for data + model persistence
3. **Upload data** to `MyDrive/ColabData/AITraining_OHLCV_H1.csv`
4. Run cells sequentially

## Output
Models saved to `MyDrive/ColabModels/` → copy to MT5 `Resources/`

## 0. Colab Environment Setup

In [ ]:
# ============================================================
# Cell 0 — Colab Environment & GPU Check
# ============================================================

import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')

# --- GPU Detection ---
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    DEVICE = 'cuda'
else:
    print("⚠️  No GPU — using CPU (slow)")
    DEVICE = 'cpu'

# --- Google Drive Mount ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = '/content/drive/MyDrive'
DATA_DIR = DRIVE_ROOT + '/ColabData'
MODEL_DIR = DRIVE_ROOT + '/ColabModels'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"\nData dir: {DATA_DIR}")
print(f"Model dir: {MODEL_DIR}")
print(f"Files in data: {os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else 'DIR NOT FOUND'}")

## 1. Install Dependencies

In [ ]:
# ============================================================
# Cell 1 — Stable ML Stack (Colab-optimized)
# ============================================================

import os
import numpy as np

# 1. Force NumPy 1.x (safe for ONNX/CatBoost)
if not np.__version__.startswith("1."):
    print(f"Detected NumPy {np.__version__}. Downgrading to 1.26.4...")
    !pip install -q --force-reinstall "numpy==1.26.4"
    print("\n✅ NumPy pinned at 1.26.4. Restarting kernel...")
    os._exit(0)

# 2. Clean up invalid NVIDIA dist warnings
!rm -rf /usr/local/lib/python3.12/dist-packages/~vidia*
!rm -rf /usr/local/lib/python3.10/dist-packages/~vidia*

# 3. Core ML stack (CUDA 12.1 build)
!pip install -q --upgrade pip
!pip install -q \
    torch==2.3.0+cu121 \
    torchvision==0.18.0+cu121 \
    torchaudio==2.3.0+cu121 \
    --index-url https://download.pytorch.org/whl/cu121

# 4. Data & ML libraries (locked to NumPy 1.x compatible builds)
!pip install -q \
    pandas scipy scikit-learn \
    onnx \
    lightgbm catboost xgboost \
    optuna matplotlib seaborn \
    tqdm pyyaml

# 5. ONNX Runtime GPU (CUDA 12 build)
!pip install -q onnxruntime-gpu==1.18.0 \
    --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

# 6. Verify imports
import numpy as np
import torch, pandas as pd
import lightgbm as lgb, catboost as cb, xgboost as xgb
import onnx
import onnxruntime as ort

print("✅ All imports OK")
print(f"   NumPy Version: {np.__version__}")
print(f"   Torch Version: {torch.__version__}")
print(f"   Torch CUDA: {torch.cuda.is_available()}")
print(f"   ONNX Runtime providers: {ort.get_available_providers()}")


## 2. Copy Project Source to Colab

In [ ]:
# ============================================================
# Cell 2 — Copy Project Source (data_pipeline, models, train_model, validate_model)
# ============================================================

# Option A: If repo is in Drive
REPO_DRIVE = DRIVE_ROOT + '/metatrader-multistrategy-ea'
if os.path.exists(REPO_DRIVE):
    import shutil
    shutil.copytree(REPO_DRIVE + '/Python', '/content/project', dirs_exist_ok=True)
    print(f"Copied from {REPO_DRIVE}/Python")
else:
    !git clone -q https://github.com/YOUR_USER/metatrader-multistrategy-ea.git /content/repo 2>/dev/null || true
    if os.path.exists('/content/repo/Python'):
        import shutil
        shutil.copytree('/content/repo/Python', '/content/project', dirs_exist_ok=True)
        print("Cloned from GitHub")
    else:
        raise FileNotFoundError("Put project in Drive at /MyDrive/metatrader-multistrategy-ea or update GitHub URL")

sys.path.insert(0, '/content/project')
print(f"Project files: {os.listdir('/content/project')}")

## 3. Configuration

In [ ]:
# ============================================================
# Cell 3 — Training Configuration
# ============================================================

from dataclasses import dataclass, field
from typing import List
from pathlib import Path

@dataclass
class Config:
    data_csv: str = DATA_DIR + '/AITraining_OHLCV_H1.csv'
    output_dir: str = MODEL_DIR
    train_pytorch: bool = True
    pytorch_models: List[str] = field(default_factory=lambda: ['patchtst', 'itransformer', 'mlp'])
    train_asset_class: bool = True
    train_deriv_families: bool = True
    train_stackers: bool = True
    seq_len: int = 60
    n_features: int = 57
    n_classes: int = 3
    tb_k: float = 1.5
    tb_vertical_bars: int = 20
    epochs: int = 80
    batch_size: int = 256
    lr: float = 3e-4
    weight_decay: float = 1e-4
    patience: int = 20
    min_ic: float = 0.02
    use_amp: bool = True
    cpcv_splits: int = 6
    family_id: int = -1
    asset_class: int = -1
    device: str = DEVICE
    num_workers: int = 2
    seed: int = 42
    force_export: bool = False

cfg = Config()

import random
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

print(f"Config:")
print(f"  Data: {cfg.data_csv}")
print(f"  Output: {cfg.output_dir}")
print(f"  Device: {cfg.device}")
print(f"  AMP: {cfg.use_amp}")
print(f"  Batch: {cfg.batch_size}, Epochs: {cfg.epochs}")
print(f"  PyTorch: {cfg.train_pytorch} ({cfg.pytorch_models})")

## 4. Load & Inspect Data

In [ ]:
# ============================================================
# Cell 4 — Load Data
# ============================================================

data_path = Path(cfg.data_csv)
if not data_path.exists():
    for name in ['AITraining_OHLCV_H1.csv', 'AITraining_OHLCV_M30.csv', 'AITraining_OHLCV_M15.csv', 'UNIVERSAL_TRAINING_SET.csv']:
        p = Path(DATA_DIR) / name
        if p.exists():
            data_path = p
            cfg.data_csv = str(p)
            break

if not data_path.exists():
    raise FileNotFoundError(f"No training CSV in {DATA_DIR}. Upload to Drive first.")

print(f"Loading: {data_path}")
df = pd.read_csv(data_path, usecols=['symbol', 'date'])
df['date'] = pd.to_datetime(df['date'])
print(f"Rows: {len(df):,}")
print(f"Symbols: {df['symbol'].nunique()}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"Symbol counts:\n{df['symbol'].value_counts().to_string()}")

## 5. PyTorch Models (GPU + AMP)

In [ ]:
# ============================================================
# Cell 5 — PyTorch Training with Mixed Precision
# ============================================================

if cfg.train_pytorch:
    from data_pipeline import build_scaled_dataset_splits, PipelineMetadata
    from models import SequenceMLP, PatchTST, iTransformer
    from train_model import train_candidate, compute_ic, export_onnx, instantiate_models, build_loader
    from validate_model import run_cpcv
    from torch.cuda.amp import autocast, GradScaler
    from torch.utils.data import DataLoader
    import torch.nn as nn
    import torch.optim as optim

    seq_len = cfg.seq_len
    if cfg.family_id in (3, 4) and seq_len == 60:
        seq_len = 120

    scaler_out = cfg.output_dir + '/scaler.bin'
    train_split, val_split, test_split, metadata = build_scaled_dataset_splits(
        cfg.data_csv, seq_len=seq_len, k=cfg.tb_k, vertical_bars=cfg.tb_vertical_bars,
        scaler_output=scaler_out, family_id=cfg.family_id, asset_class=cfg.asset_class
    )

    print(f"Dataset: train={metadata.train_size:,}, val={metadata.val_size:,}, test={metadata.test_size:,}, feat={metadata.n_features}")

    def train_candidate_amp(name, model, train_split, val_split, test_split, metadata, cfg):
        train_loader = build_loader(train_split, cfg.batch_size, True)
        val_loader = build_loader(val_split, cfg.batch_size, False)
        test_loader = build_loader(test_split, cfg.batch_size, False)

        y_tr = train_split[1]
        counts = np.bincount(y_tr, minlength=3)
        class_w = torch.tensor(1.0 / np.maximum(counts, 1), device=cfg.device, dtype=torch.float32)
        criterion = nn.CrossEntropyLoss(weight=class_w, reduction='none')

        model = model.to(cfg.device)
        opt = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        total_steps = cfg.epochs * max(1, len(train_loader))
        sched = optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=total_steps, pct_start=0.15)
        scaler = GradScaler(enabled=cfg.use_amp)

        best_ic, best_state, patience = -1e9, None, 0
        for ep in range(cfg.epochs):
            model.train()
            total_loss = 0.0
            for x, y, w, _ in train_loader:
                x, y, w = x.to(cfg.device), y.to(cfg.device), w.to(cfg.device)
                opt.zero_grad()
                with autocast(enabled=cfg.use_amp):
                    loss = (criterion(model(x), y) * w).mean()
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                total_loss += loss.item()

            model.eval()
            scores, rets = [], []
            with torch.no_grad(), autocast(enabled=cfg.use_amp):
                for x, _y, _w, ret in val_loader:
                    probs = torch.softmax(model(x.to(cfg.device)), -1).cpu().numpy()
                    scores.extend((probs[:, 2] - probs[:, 0]).tolist())
                    rets.extend(ret.numpy().tolist())
            from scipy.stats import spearmanr
            val_ic = float(spearmanr(scores, rets)[0]) if len(scores) > 10 else 0.0

            if val_ic > best_ic:
                best_ic, best_state, patience = val_ic, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
            else:
                patience += 1

            if (ep + 1) % 10 == 0 or ep == 0:
                print(f"  ep={ep+1:03d} loss={total_loss/len(train_loader):.5f} val_ic={val_ic:.4f} best={best_ic:.4f}")
            if patience >= cfg.patience:
                print(f"  early stop at ep={ep+1}")
                break

        if best_state:
            model.load_state_dict(best_state)
        test_ic = compute_ic(model, test_loader, cfg.device)
        print(f"  final: val_ic={best_ic:.4f} test_ic={test_ic:.4f} ann={metadata.annualization:.1f}")
        return best_ic, test_ic, model

    results = {}
    for mtype in cfg.pytorch_models:
        print(f"\n{'='*50}")
        print(f"TRAINING: {mtype.upper()}")
        print(f"{'='*50}")

        model = instantiate_models(mtype, metadata.seq_len, metadata.n_features)[mtype]
        print(f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

        val_ic, test_ic, trained = train_candidate_amp(mtype, model, train_split, val_split, test_split, metadata, cfg)

        tmp = cfg.output_dir + '/' + mtype + '.tmp.onnx'
        export_onnx(trained, metadata.seq_len, metadata.n_features, tmp, opset=12)

        X_all = np.concatenate([train_split[0], val_split[0], test_split[0]])
        y_all = np.concatenate([train_split[1], val_split[1], test_split[1]])
        r_all = np.concatenate([train_split[3], val_split[3], test_split[3]])
        cpcv = run_cpcv(tmp, X_all, y_all, r_all, cfg.cpcv_splits, metadata.annualization)

        ok = test_ic >= cfg.min_ic and cpcv['deploy_gate']
        if ok or cfg.force_export:
            Path(tmp).rename(cfg.output_dir + '/' + mtype + '.onnx')
            print(f"  ✅ Exported {mtype}.onnx")
        else:
            Path(tmp).unlink(missing_ok=True)
            print(f"  ❌ Gate failed")

        results[mtype] = {'val_ic': val_ic, 'test_ic': test_ic, 'cpcv': cpcv, 'ok': ok}

    print(f"\n{'='*50}")
    print("PYTORCH SUMMARY")
    for k, v in results.items():
        print(f"  {k:12s} val_ic={v['val_ic']:.4f} test_ic={v['test_ic']:.4f} cpcv_sharpe={v['cpcv']['mean_sr']:.4f} {'✅' if v['ok'] else '❌'}")

    with open(cfg.output_dir + '/pytorch_results.json', 'w') as f:
        json.dump({k: {kk: float(vv) if isinstance(vv, (np.floating, np.integer)) else vv for kk, vv in v.items() if kk != 'cpcv'} for k, v in results.items()}, f, indent=2)

else:
    print("PyTorch training skipped")
    results = {}

## 6. Asset Class Models (Tree-Based)

In [ ]:
# ============================================================
# Cell 6 — Asset Class Models
# ============================================================

asset_results = {}

if cfg.train_asset_class:
    import pickle
    from data_pipeline import build_scaled_dataset_splits
    from scipy.stats import spearmanr

    asset_configs = [
        {'id': 0, 'name': 'forex', 'models': ['lgbm']},
        {'id': 1, 'name': 'metals', 'models': ['catboost', 'xgboost']},
        {'id': 2, 'name': 'indices', 'models': ['xgboost']},
        {'id': 3, 'name': 'energies', 'models': ['xgboost']},
    ]

    for ac in asset_configs:
        print(f"\n{'='*50}")
        print(f"{ac['name'].upper()} (asset_class={ac['id']})")
        print(f"{'='*50}")

        tr, va, te, meta = build_scaled_dataset_splits(
            cfg.data_csv, seq_len=cfg.seq_len, k=cfg.tb_k, vertical_bars=cfg.tb_vertical_bars,
            family_id=-1, asset_class=ac['id']
        )
        print(f"  Data: {meta.train_size:,}/{meta.val_size:,}/{meta.test_size:,} feat={meta.n_features}")

        X_tr = tr[0].reshape(len(tr[0]), -1)
        X_va = va[0].reshape(len(va[0]), -1)
        X_te = te[0].reshape(len(te[0]), -1)
        y_tr, y_va, y_te = tr[1], va[1], te[1]
        r_te = te[3]

        ac_res = {}
        for mt in ac['models']:
            print(f"  Training {mt.upper()}...")

            if mt == 'lgbm':
                import lightgbm as lgb
                params = {
                    'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
                    'learning_rate': 0.025 if ac['name']=='forex' else 0.03,
                    'num_leaves': 31 if ac['name']=='forex' else 63,
                    'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5,
                    'lambda_l1': 0.1, 'lambda_l2': 0.1, 'min_child_samples': 50, 'verbose': -1
                }
                model = lgb.train(params, lgb.Dataset(X_tr, label=y_tr),
                               valid_sets=[lgb.Dataset(X_va, label=y_va)],
                               num_boost_round=800 if ac['name']=='forex' else 1000,
                               callbacks=[lgb.early_stopping(50)])
                preds = model.predict(X_te)
            elif mt == 'catboost':
                import catboost as cb
                model = cb.CatBoostClassifier(
                    iterations=600 if ac['name']=='metals' else 1000,
                    learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
                    loss_function='MultiClass', classes_count=3,
                    verbose=0, early_stopping_rounds=50
                )
                model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
                preds = model.predict_proba(X_te)
            elif mt == 'xgboost':
                import xgboost as xgb
                if ac['name'] == 'metals':
                    model = xgb.XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6,
                        subsample=0.8, colsample_bytree=0.7, objective='multi:softprob',
                        num_class=3, eval_metric='mlogloss', early_stopping_rounds=50, verbosity=0)
                elif ac['name'] == 'indices':
                    model = xgb.XGBClassifier(n_estimators=600, learning_rate=0.025, max_depth=5,
                        subsample=0.8, colsample_bytree=0.7, objective='multi:softprob',
                        num_class=3, eval_metric='mlogloss', early_stopping_rounds=50, verbosity=0)
                else:
                    model = xgb.XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6,
                        subsample=0.8, colsample_bytree=0.7, objective='multi:softprob',
                        num_class=3, eval_metric='mlogloss', early_stopping_rounds=50, verbosity=0)
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
                preds = model.predict_proba(X_te)

            ic, _ = spearmanr(preds[:, 2] - preds[:, 0], r_te)
            out_path = cfg.output_dir + '/' + ac['name'] + '_' + mt + '.pkl'
            with open(out_path, 'wb') as f:
                pickle.dump(model, f)
            ac_res[mt] = {'test_ic': float(ic), 'path': out_path}
            print(f"    IC={ic:.4f} → {out_path}")

        asset_results[ac['name']] = ac_res

    with open(cfg.output_dir + '/asset_class_results.json', 'w') as f:
        json.dump(asset_results, f, indent=2)
    print("\n✅ Asset class models done")
else:
    print("Asset class training skipped")

## 7. Deriv Family Models (18 Families)

In [ ]:
# ============================================================
# Cell 7 — Deriv Family Models
# ============================================================

FAMILY_NAMES = [
    'crashboom', 'volatility', 'step', 'jump', 'dex',
    'multistep', 'exponential', 'hybrid', 'rangebreak',
    'skewstep', 'volswitch', 'driftswitch', 'trek',
    'tactical', 'derived', 'stablespread', 'pairsarbitrage', 'spotvolatility'
]

FAMILY_MODELS = {
    0: ['catboost'], 1: ['lgbm'], 2: ['xgboost'], 3: ['catboost', 'lgbm'], 4: ['catboost', 'lgbm'],
    5: ['xgboost'], 6: ['catboost'], 7: ['catboost', 'lgbm'], 8: ['xgboost'], 9: ['xgboost'],
    10: ['catboost'], 11: ['catboost'], 12: ['xgboost'], 13: ['catboost'],
    14: ['xgboost'], 15: ['catboost'], 16: ['catboost'], 17: ['xgboost']
}

deriv_results = {}

if cfg.train_deriv_families:
    import pickle
    from data_pipeline import build_scaled_dataset_splits
    from scipy.stats import spearmanr
    from catboost import CatBoostClassifier, Pool

    for fid in range(18):
        fname = FAMILY_NAMES[fid]
        models = FAMILY_MODELS.get(fid, ['catboost'])
        sl = cfg.seq_len if not (fid in (3, 4) and cfg.seq_len == 60) else 120

        print(f"\n{'='*50}")
        print(f"FAMILY {fid:2d}: {fname.upper()} (seq={sl})")
        print(f"{'='*50}")

        tr, va, te, meta = build_scaled_dataset_splits(
            cfg.data_csv, seq_len=sl, k=cfg.tb_k, vertical_bars=cfg.tb_vertical_bars, family_id=fid
        )
        print(f"  Data: {meta.train_size:,}/{meta.val_size:,}/{meta.test_size:,} feat={meta.n_features}")

        if meta.train_size < 100:
            print(f"  ⚠️  Skipping (insufficient data)")
            continue

        X_tr = tr[0].reshape(len(tr[0]), -1)
        X_va = va[0].reshape(len(va[0]), -1)
        X_te = te[0].reshape(len(te[0]), -1)
        y_tr, y_va, y_te = tr[1], va[1], te[1]
        r_te = te[3]

        fam_res = {}
        for mt in models:
            print(f"  {mt.upper()}...")

            if mt == 'catboost':
                if fid in (0, 4):
                    m = CatBoostClassifier(iterations=1500, learning_rate=0.025, depth=8,
                        loss_function='MultiClass', classes_count=3, l2_leaf_reg=5.0,
                        random_seed=42, verbose=100, early_stopping_rounds=75,
                        class_weights=[1.0, 0.5, 1.0])
                elif fid == 7:
                    m = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=7,
                        loss_function='MultiClass', classes_count=3, l2_leaf_reg=4.0,
                        random_seed=42, verbose=100, early_stopping_rounds=50)
                else:
                    m = CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=6,
                        loss_function='MultiClass', classes_count=3, l2_leaf_reg=3.0,
                        random_seed=42, verbose=100, early_stopping_rounds=50)
                m.fit(Pool(X_tr, label=y_tr), eval_set=Pool(X_va, label=y_va))
                preds = m.predict_proba(X_te)

            elif mt == 'lgbm':
                import lightgbm as lgb
                if fid == 1:
                    params = {'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
                        'learning_rate': 0.02, 'num_leaves': 31, 'feature_fraction': 0.7,
                        'bagging_fraction': 0.8, 'bagging_freq': 5, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
                        'min_child_samples': 50, 'verbose': -1}
                else:
                    params = {'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
                        'learning_rate': 0.03, 'num_leaves': 63, 'feature_fraction': 0.7,
                        'bagging_fraction': 0.8, 'bagging_freq': 5, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
                        'min_child_samples': 50, 'verbose': -1}
                m = lgb.train(params, lgb.Dataset(X_tr, label=y_tr),
                           valid_sets=[lgb.Dataset(X_va, label=y_va)],
                           num_boost_round=1000, callbacks=[lgb.early_stopping(50)])
                preds = m.predict(X_te)

            elif mt == 'xgboost':
                import xgboost as xgb
                if fid in (2, 5, 9):
                    m = xgb.XGBClassifier(objective='multi:softprob', num_class=3, learning_rate=0.03,
                        max_depth=6, n_estimators=1000, subsample=0.8, colsample_bytree=0.8,
                        gamma=1.0, reg_alpha=0.5, reg_lambda=2.0, min_child_weight=50,
                        early_stopping_rounds=50)
                else:
                    m = xgb.XGBClassifier(objective='multi:softprob', num_class=3, learning_rate=0.03,
                        max_depth=6, n_estimators=1000, subsample=0.8, colsample_bytree=0.7,
                        reg_alpha=0.1, reg_lambda=0.1, min_child_weight=50,
                        early_stopping_rounds=50, verbosity=0)
                m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=100)
                preds = m.predict_proba(X_te)

            ic, _ = spearmanr(preds[:, 2] - preds[:, 0], r_te)
            out = cfg.output_dir + '/' + fname + '_' + mt + '.pkl'
            with open(out, 'wb') as f:
                pickle.dump(m, f)
            fam_res[mt] = {'test_ic': float(ic), 'path': out}
            print(f"    IC={ic:.4f}")

        deriv_results[fname] = {'family_id': fid, 'seq_len': sl, 'models': fam_res, 'n': meta.train_size}

    with open(cfg.output_dir + '/deriv_family_results.json', 'w') as f:
        json.dump(deriv_results, f, indent=2)
    print("\n✅ Deriv family models done")
else:
    print("Deriv family training skipped")

## 8. Stackers (Ridge on OOF)

In [ ]:
# ============================================================
# Cell 8 — Family Stackers
# ============================================================

stacker_results = {}

if cfg.train_stackers:
    import pickle
    import onnxruntime as ort
    from scipy.special import softmax
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import TimeSeriesSplit
    from data_pipeline import build_scaled_dataset_splits
    from scipy.stats import spearmanr

    for fid in range(18):
        fname = FAMILY_NAMES[fid]
        sl = cfg.seq_len if not (fid in (3, 4) and cfg.seq_len == 60) else 120

        onnx_p = cfg.output_dir + '/patchtst.onnx'
        lgbm_p = cfg.output_dir + '/' + fname + '_lgbm.pkl'
        cb_p = cfg.output_dir + '/' + fname + '_catboost.pkl'
        xgb_p = cfg.output_dir + '/' + fname + '_xgboost.pkl'

        if not (os.path.exists(onnx_p) and os.path.exists(lgbm_p)):
            print(f"{fname}: missing base models (need ONNX + LGBM min)")
            continue

        print(f"\nStacker: {fname}")
        tr, va, te, _ = build_scaled_dataset_splits(
            cfg.data_csv, seq_len=sl, k=cfg.tb_k, vertical_bars=cfg.tb_vertical_bars, family_id=fid
        )
        X_all = np.concatenate([tr[0], va[0]]); y_all = np.concatenate([tr[1], va[1]]); r_all = np.concatenate([tr[3], va[3]])
        X_te, r_te = te[0], te[3]
        X_all_f = X_all.reshape(len(X_all), -1); X_te_f = X_te.reshape(len(X_te), -1)

        sess = ort.InferenceSession(onnx_p)
        iname = sess.get_inputs()[0].name
        onnx_all = softmax(sess.run(None, {iname: X_all.astype(np.float32)})[0], axis=1)
        onnx_te = softmax(sess.run(None, {iname: X_te.astype(np.float32)})[0], axis=1)

        with open(lgbm_p, 'rb') as f: lgbm = pickle.load(f)
        lgbm_all = lgbm.predict(X_all_f); lgbm_te = lgbm.predict(X_te_f)

        cb_all = cb_te = xgb_all = xgb_te = None
        if os.path.exists(cb_p):
            with open(cb_p, 'rb') as f: cb = pickle.load(f)
            cb_all = cb.predict_proba(X_all_f); cb_te = cb.predict_proba(X_te_f)
        if os.path.exists(xgb_p):
            with open(xgb_p, 'rb') as f: xgb = pickle.load(f)
            xgb_all = xgb.predict_proba(X_all_f); xgb_te = xgb.predict_proba(X_te_f)

        n_base = 2 + (cb_all is not None) + (xgb_all is not None)
        meta_X = np.zeros((len(X_all), n_base * 3), np.float32)
        tscv = TimeSeriesSplit(n_splits=5, gap=20)
        for _, vi in tscv.split(X_all):
            c = 0
            meta_X[vi, c:c+3] = onnx_all[vi]; c += 3
            meta_X[vi, c:c+3] = lgbm_all[vi]; c += 3
            if cb_all is not None: meta_X[vi, c:c+3] = cb_all[vi]; c += 3
            if xgb_all is not None: meta_X[vi, c:c+3] = xgb_all[vi]; c += 3

        scaler = StandardScaler().fit(meta_X)
        ridge = Ridge(alpha=1.0).fit(scaler.transform(meta_X), (y_all == 2).astype(float))

        parts = [onnx_te, lgbm_te]
        if cb_te is not None: parts.append(cb_te)
        if xgb_te is not None: parts.append(xgb_te)
        stack_sig = ridge.predict(scaler.transform(np.hstack(parts)))

        bundle = {'ridge': ridge, 'scaler': scaler, 'family_id': fid, 'n_base_models': n_base}
        out = cfg.output_dir + '/' + fname + '_stacker.pkl'
        with open(out, 'wb') as f: pickle.dump(bundle, f)

        print(f"  ONNX IC: {spearmanr(onnx_te[:,2]-onnx_te[:,0], r_te)[0]:.4f}")
        print(f"  LGBM IC: {spearmanr(lgbm_te[:,2]-lgbm_te[:,0], r_te)[0]:.4f}")
        if cb_te is not None: print(f"  CatB IC: {spearmanr(cb_te[:,2]-cb_te[:,0], r_te)[0]:.4f}")
        if xgb_te is not None: print(f"  XGB IC:  {spearmanr(xgb_te[:,2]-xgb_te[:,0], r_te)[0]:.4f}")
        print(f"  Stack  IC: {spearmanr(stack_sig, r_te)[0]:.4f}")
        print(f"  Saved: {out}")

        stacker_results[fname] = {'family_id': fid, 'n_base': n_base, 'path': out,
            'test_ic_onnx': float(spearmanr(onnx_te[:,2]-onnx_te[:,0], r_te)[0]),
            'test_ic_lgbm': float(spearmanr(lgbm_te[:,2]-lgbm_te[:,0], r_te)[0]),
            'test_ic_stack': float(spearmanr(stack_sig, r_te)[0])}

    with open(cfg.output_dir + '/stacker_results.json', 'w') as f:
        json.dump(stacker_results, f, indent=2)
    print("\n✅ Stackers done")
else:
    print("Stacker training skipped")

## 9. Universal Stacker (ONNX + All Asset Models)

In [ ]:
# ============================================================
# Cell 9 — Universal Stacker
# ============================================================

import onnxruntime as ort
from scipy.special import softmax
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from data_pipeline import build_scaled_dataset_splits
from scipy.stats import spearmanr
import pickle

onnx_p = cfg.output_dir + '/patchtst.onnx'
if os.path.exists(onnx_p):
    print("Training universal stacker...")
    tr, va, te, _ = build_scaled_dataset_splits(
        cfg.data_csv, seq_len=cfg.seq_len, k=cfg.tb_k, vertical_bars=cfg.tb_vertical_bars,
        family_id=-1, asset_class=-1
    )
    X_all = np.concatenate([tr[0], va[0]]); y_all = np.concatenate([tr[1], va[1]]); r_all = np.concatenate([tr[3], va[3]])
    X_te, r_te = te[0], te[3]
    X_all_f = X_all.reshape(len(X_all), -1); X_te_f = X_te.reshape(len(X_te), -1)

    sess = ort.InferenceSession(onnx_p)
    iname = sess.get_inputs()[0].name
    onnx_all = softmax(sess.run(None, {iname: X_all.astype(np.float32)})[0], axis=1)
    onnx_te = softmax(sess.run(None, {iname: X_te.astype(np.float32)})[0], axis=1)

    asset_models = {}
    for ac in ['forex', 'metals', 'indices', 'energies']:
        for mt in ['lgbm', 'catboost', 'xgboost']:
            p = cfg.output_dir + '/' + ac + '_' + mt + '.pkl'
            if os.path.exists(p):
                with open(p, 'rb') as f: asset_models[ac + '_' + mt] = pickle.load(f)

    print(f"Loaded {len(asset_models)} asset models")

    asset_all, asset_te = {}, {}
    for name, m in asset_models.items():
        if hasattr(m, 'predict_proba'):
            asset_all[name] = m.predict_proba(X_all_f)
            asset_te[name] = m.predict_proba(X_te_f)
        else:
            asset_all[name] = m.predict(X_all_f)
            asset_te[name] = m.predict(X_te_f)

    n_base = 1 + len(asset_models)
    meta_X = np.zeros((len(X_all), n_base * 3), np.float32)
    tscv = TimeSeriesSplit(n_splits=5, gap=20)
    for _, vi in tscv.split(X_all):
        c = 0
        meta_X[vi, c:c+3] = onnx_all[vi]; c += 3
        for name in asset_models:
            meta_X[vi, c:c+3] = asset_all[name][vi]; c += 3

    scaler = StandardScaler().fit(meta_X)
    ridge = Ridge(alpha=1.0).fit(scaler.transform(meta_X), (y_all == 2).astype(float))

    parts = [onnx_te] + [asset_te[n] for n in asset_models]
    stack_sig = ridge.predict(scaler.transform(np.hstack(parts)))

    bundle = {'ridge': ridge, 'scaler': scaler, 'model_names': ['onnx'] + list(asset_models.keys()), 'n_base_models': n_base}
    with open(cfg.output_dir + '/universal_stacker.pkl', 'wb') as f: pickle.dump(bundle, f)

    print(f"  ONNX IC: {spearmanr(onnx_te[:,2]-onnx_te[:,0], r_te)[0]:.4f}")
    for n in asset_models:
        ic = spearmanr(asset_te[n][:,2]-asset_te[n][:,0], r_te)[0]
        print(f"  {n:20s} IC: {ic:.4f}")
    print(f"  Stack IC: {spearmanr(stack_sig, r_te)[0]:.4f}")
    print(f"  Saved: universal_stacker.pkl")
else:
    print("Universal ONNX not found — skipping universal stacker")

## 10. Final Summary & Model Registry

In [ ]:
# ============================================================
# Cell 10 — Summary & Registry
# ============================================================

print("="*70)
print("TRAINING COMPLETE — MODEL REGISTRY")
print("="*70)

files = list(Path(cfg.output_dir).glob('*.onnx')) + list(Path(cfg.output_dir).glob('*.pkl')) + list(Path(cfg.output_dir).glob('*.bin'))
print(f"\nOutput: {cfg.output_dir}")
print(f"Files: {len(files)}")
for f in sorted(files):
    print(f"  {f.name:45s} {f.stat().st_size/1e6:>7.2f} MB")

registry = {'timestamp': pd.Timestamp.now().isoformat(), 'data_file': cfg.data_csv,
    'config': {'seq_len': cfg.seq_len, 'tb_k': cfg.tb_k, 'epochs': cfg.epochs, 'batch_size': cfg.batch_size, 'device': cfg.device},
    'models': {}}

if 'results' in locals():
    for k, v in results.items():
        p = cfg.output_dir + '/' + k + '.onnx'
        if os.path.exists(p):
            registry['models']['pytorch_' + k] = {'type': 'onnx', 'path': p, 'test_ic': v['test_ic'], 'cpcv_sharpe': v['cpcv']['mean_sr'], 'deployed': v['ok']}

if 'asset_results' in locals():
    for ac, ms in asset_results.items():
        for mt, r in ms.items():
            registry['models']['asset_' + ac + '_' + mt] = {'type': 'tree', 'asset': ac, 'model': mt, 'path': r['path'], 'test_ic': r['test_ic']}

if 'deriv_results' in locals():
    for fn, r in deriv_results.items():
        for mt, m in r['models'].items():
            registry['models']['deriv_' + fn + '_' + mt] = {'type': 'tree', 'family': fn, 'model': mt, 'path': m['path'], 'test_ic': m['test_ic'], 'seq_len': r['seq_len']}

if 'stacker_results' in locals():
    for fn, r in stacker_results.items():
        registry['models']['stacker_' + fn] = {'type': 'stacker', 'family': fn, 'path': r['path'], 'n_base': r['n_base'], 'test_ic_stack': r['test_ic_stack']}

if os.path.exists(cfg.output_dir + '/universal_stacker.pkl'):
    registry['models']['universal_stacker'] = {'type': 'stacker', 'scope': 'universal', 'path': cfg.output_dir + '/universal_stacker.pkl'}

reg_path = cfg.output_dir + '/model_registry.json'
with open(reg_path, 'w') as f:
    json.dump(registry, f, indent=2, default=str)

print(f"\nRegistry: {reg_path} ({len(registry['models'])} models)")
print("="*70)

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(cfg.output_dir + '/patchtst.onnx')
    dummy = np.random.randn(1, 60, 57).astype(np.float32)
    out = sess.run(None, {sess.get_inputs()[0].name: dummy})[0]
    print(f"\n✅ ONNX inference OK: {out.shape} → class {np.argmax(out[0])}")
except Exception as e:
    print(f"\n⚠️  ONNX test failed: {e}")

## 11. Download Models (Optional — already in Drive)

In [ ]:
# Models are already saved to Google Drive at:
# /content/drive/MyDrive/ColabModels/

# To download locally, you can:
# 1. Right-click folder in Drive UI → Download
# 2. Or use gsutil/gdrive CLI
# 3. Or zip in Colab:

# !zip -r /content/models.zip /content/drive/MyDrive/ColabModels
# from google.colab import files
# files.download('/content/models.zip')

print(f"Models ready at: {MODEL_DIR}")
print("Copy .onnx + scaler.bin to MT5 Resources/ folder")
print("Copy .pkl stackers to Python/ for inference service")